# 02 — Silver Transformation

## Objective

Transform the raw Bronze table into a
clean, typed table suitable for exploratory analysis and model training.

The Bronze layer intentionally stored all 115 feature columns as `string`, matching
the raw source format with no transformation applied. This notebook performs the
first transformation step of the Medallion architecture: casting those columns to
their correct numeric type (`double`) and validating that the cast did not silently
introduce data quality issues.



In [0]:
# Create the silver_layer schema if it doesn't already exist
spark.sql("CREATE SCHEMA IF NOT EXISTS kitsune_project.silver_layer")

#Read the bronze table into a DataFrame
bronze_df = spark.table("kitsune_project.bronze_layer.syn_dos_raw")

# Sanity check: confirm row and column counts match what was validated in 01_bronze_ingestion
print(f"Row count: {bronze_df.count()}")
print(f"Column count: {len(bronze_df.columns)}")

In [0]:
from pyspark.sql import functions as F

# Derive feature columns from the actual schema instead of hardcoding a count
feature_columns = [c for c in bronze_df.columns if c.startswith("feature_")]

print(f"Detected {len(feature_columns)} feature columns")

# Select row_id and label unchanged, cast every feature column to double
silver_df = bronze_df.select(
    "row_id",
    F.col("label").cast("int").alias("label"),
    *[F.col(c).cast("double").alias(c) for c in feature_columns]
)

# Confirm the schema actually changed 
silver_df.printSchema()

In [0]:
bronze_nulls = bronze_df.select(
    F.sum(F.col("label").isNull().cast("int")).alias("label"),
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in feature_columns]
).withColumn("source", F.lit("bronze"))

silver_nulls = silver_df.select(
    F.sum(F.col("label").isNull().cast("int")).alias("label"),
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in feature_columns]
).withColumn("source", F.lit("silver"))

#Number of null values in bronze and silver
display(bronze_nulls.union(silver_nulls).select("source", "label", *feature_columns))

# Sum across all columns (label + features) to get one total per layer
bronze_total_nulls = bronze_nulls.drop("source").select(
    sum([F.col(c) for c in ["label"] + feature_columns]).alias("total_nulls")
).collect()[0]["total_nulls"]

silver_total_nulls = silver_nulls.drop("source").select(
    sum([F.col(c) for c in ["label"] + feature_columns]).alias("total_nulls")
).collect()[0]["total_nulls"]

print(f"Bronze total nulls (label + 115 features): {bronze_total_nulls}")
print(f"Silver total nulls (label + 115 features): {silver_total_nulls}")

In [0]:
#Check the label values
label_values = silver_df.select("label").distinct().collect()

# Print the label values
print(label_values)

In [0]:
#Check the label counts
silver_df.groupBy("label").count().show()

We have an imbalanced DataFrame.

In [0]:
#Check the number of rows and distinct row_ids
total_rows = silver_df.count()
distinct_ids = silver_df.select("row_id").distinct().count()

print(f"Total rows: {total_rows}")
print(f"Distinct row_ids: {distinct_ids}")

In [0]:
#Save the silver table
silver_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("kitsune_project.silver_layer.syn_dos_clean")

## Silver Transformation Summary

### What We Did

**Input:** Bronze table `kitsune_project.bronze_layer.syn_dos_raw`
* 2,771,276 rows
* 117 columns (row_id + label + 115 features)
* All features stored as `string` (raw format)

**Transformations Applied:**
1. **Schema Creation** — Created `kitsune_project.silver_layer` schema
2. **Type Casting:**
   * `label`: string → int (binary: 0=normal, 1=attack)
   * `feature_1` through `feature_115`: string → double
   * `row_id`: unchanged (long)
3. **Data Quality Validation:**
   * Verified row/column counts match Bronze layer
   * Checked for null values before and after casting
   * Confirmed zero cast failures (no data loss)
   * Verified label domain (only 0/1, no unexpected values)
   * Verified no duplicate row_ids
   * Checked class distribution

**Output:** Silver table `kitsune_project.silver_layer.syn_dos_clean`
* 2,771,276 rows (100% retention)
* 117 columns with proper numeric types
* 0 null values across all columns
* Ready for exploratory analysis and ML model training

### Key Validations Passed
✓ Row count preserved: 2,771,276  
✓ Column count preserved: 117  
✓ Zero nulls in Bronze: 0  
✓ Zero nulls in Silver: 0  
✓ No cast failures detected  
✓ Label domain verified: only {0, 1}  
✓ No duplicate row_ids: 2,771,276 total rows, 2,771,276 distinct row_ids  
✓ Class distribution checked: 2,764,238 normal (99.75%), 7,038 attack (0.25%)
  — severe class imbalance, to be addressed in Gold layer (resampling or
  class weighting strategy)

### Next Steps
The Silver table is now ready for:
* Exploratory Data Analysis (EDA)
* Feature engineering
* Model training and evaluation, with special attention to the class
  imbalance identified above (accuracy alone will be a misleading metric —
  precision, recall, and F1 on the minority class will matter more)